In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np

sns.set(style="whitegrid")
train = pd.read_csv("/kaggle/input/hms-harmful-brain-activity-classification/train.csv")
test = pd.read_csv("/kaggle/input/hms-harmful-brain-activity-classification/test.csv")
all_df = pd.concat([train, test]).reset_index(drop=True)

try:
    display(train.head())
    display(test.head())
    display(all_df.head())
except NameError:
    print(train.head())
    print(test.head())
    print(all_df.head())



,eeg_id,eeg_sub_id,eeg_label_offset_seconds,spectrogram_id,spectrogram_sub_id,spectrogram_label_offset_seconds,label_id,patient_id,expert_consensus,seizure_vote,lpd_vote,gpd_vote,lrda_vote,grda_vote,other_vote
0,1628180742,0,0.0,353733,0,0.0,127492639,42516,Seizure,3,0,0,0,0,0
1,1628180742,1,6.0,353733,1,6.0,3887563113,42516,Seizure,3,0,0,0,0,0
2,1628180742,2,8.0,353733,2,8.0,1142670488,42516,Seizure,3,0,0,0,0,0
3,1628180742,3,18.0,353733,3,18.0,2718991173,42516,Seizure,3,0,0,0,0,0
4,1628180742,4,24.0,353733,4,24.0,3080632009,42516,Seizure,3,0,0,0,0,0


,spectrogram_id,eeg_id,patient_id
0,2207717,2578018731,34153
1,2207717,2578018731,34153
2,2207717,2578018731,34153
3,2207717,2578018731,34153
4,8352559,1706196508,37552


,eeg_id,eeg_sub_id,eeg_label_offset_seconds,spectrogram_id,spectrogram_sub_id,spectrogram_label_offset_seconds,label_id,patient_id,expert_consensus,seizure_vote,lpd_vote,gpd_vote,lrda_vote,grda_vote,other_vote
0,1628180742,0.0,0.0,353733,0.0,0.0,1.274926e+08,42516,Seizure,3.0,0.0,0.0,0.0,0.0,0.0
1,1628180742,1.0,6.0,353733,1.0,6.0,3.887563e+09,42516,Seizure,3.0,0.0,0.0,0.0,0.0,0.0
2,1628180742,2.0,8.0,353733,2.0,8.0,1.142670e+09,42516,Seizure,3.0,0.0,0.0,0.0,0.0,0.0
3,1628180742,3.0,18.0,353733,3.0,18.0,2.718991e+09,42516,Seizure,3.0,0.0,0.0,0.0,0.0,0.0
4,1628180742,4.0,24.0,353733,4.0,24.0,3.080632e+09,42516,Seizure,3.0,0.0,0.0,0.0,0.0,0.0


In [2]:
print(len(train.eeg_id.unique()))
print(len(train.spectrogram_id.unique()))
print(len(train.patient_id.unique()))
print(len(train.eeg_id.unique()) / len(train.patient_id.unique()))
print(len(train.spectrogram_id.unique()) / len(train.patient_id.unique()))



15396
10024
1910
8.060732984293194
5.2481675392670155


In [3]:
train_spectrogram_dir = (
    "/kaggle/input/hms-harmful-brain-activity-classification/train_spectrograms/"
)
test_spectrogram_dir = (
    "/kaggle/input/hms-harmful-brain-activity-classification/test_spectrograms/"
)
train_eeg_dir = "/kaggle/input/hms-harmful-brain-activity-classification/train_eegs/"
test_eeg_dir = "/kaggle/input/hms-harmful-brain-activity-classification/test_eegs/"

train_spectrogram_files = []
test_spectrogram_files = []
train_eeg_files = []
test_eeg_files = []

print(
    "Skipped listing parquet directories (not needed for this constant-prior submission)."
)




Skipped listing parquet directories (not needed for this constant-prior submission).


In [4]:
def get_files_info(files, file_dir):
    nan_ratio = []
    shapes = []
    for file in files:
        data = np.array(pd.read_parquet(f"{file_dir}{file}"))
        nan_ratio.append(np.isnan(data).sum() / len(data.flatten()))
        shapes.append(data.shape)
    nan_ratio = np.array(nan_ratio)
    shapes = np.array(shapes)
    return nan_ratio, shapes




In [5]:
hypothesis0 = False
hypothesis1 = False

if len(train.spectrogram_id.unique()) / len(train.patient_id.unique()) < 5:
    hypothesis0 = True

if len(all_df.eeg_id.unique()) == (
    len(train.eeg_id.unique()) + len(test.eeg_id.unique())
):
    if len(all_df.patient_id.unique()) == (
        len(train.patient_id.unique()) + len(test.patient_id.unique())
    ):
        if len(all_df.spectrogram_id.unique()) == (
            len(train.spectrogram_id.unique()) + len(test.spectrogram_id.unique())
        ):
            hypothesis1 = True

print(f"hypothesis0: {hypothesis0}")
print(f"hypothesis1: {hypothesis1}")



hypothesis0: False
hypothesis1: False


In [6]:
hypotheses = []
hypotheses.append(len(test.eeg_id.unique()) == len(test))
hypotheses.append(len(test.spectrogram_id.unique()) == len(test))
hypotheses.append(len(test.patient_id.unique()) != len(test))
hypotheses.append(True)  # file count checks skipped (see cell 3)
hypotheses.append(True)  # file count checks skipped (see cell 3)
hypotheses.append(
    len(train.spectrogram_id.unique()) / len(train.patient_id.unique()) < 6
)
hypotheses.append(
    len(train.spectrogram_id.unique()) / len(train.patient_id.unique()) > 4
)

print(f"hypotheses: {hypotheses}")
hypotheses = all(hypotheses)
print(f"hyposetheses: {hypotheses}")



hypotheses: [False, False, True, True, True, True, True]
hyposetheses: False


In [7]:
sub = pd.read_csv(
    "/kaggle/input/hms-harmful-brain-activity-classification/sample_submission.csv"
)

targets = [
    "seizure_vote",
    "lpd_vote",
    "gpd_vote",
    "lrda_vote",
    "grda_vote",
    "other_vote",
]

# Change (score-targeted, minimal): estimate the single constant probability vector by
# averaging per-row vote *distributions* (each row normalized to sum to 1), which better
# matches the competition's distributional targets and avoids overweighting EEGs with
# many overlapping segments. Still "one vector for all rows" (core logic unchanged).
train_votes = train[targets].copy()
for c in targets:
    train_votes[c] = pd.to_numeric(train_votes[c], errors="coerce").fillna(0.0)

mat = train_votes.to_numpy(dtype=float)
mat = np.nan_to_num(mat, nan=0.0, posinf=0.0, neginf=0.0)
mat = np.clip(mat, 0.0, None)

row_sums = mat.sum(axis=1, keepdims=True)
# If any row is all zeros (shouldn't happen), fall back to uniform for that row.
row_probs = np.divide(
    mat,
    np.where(row_sums > 0.0, row_sums, 1.0),
    out=np.full_like(mat, 1.0 / len(targets), dtype=float),
    where=row_sums > 0.0,
)

p_hat = row_probs.mean(axis=0)  # average row-wise distributions

# Change (score-targeted, minimal): apply Jeffreys prior as a Dirichlet posterior mean
# using an effective sample size equal to the number of rows contributing to p_hat.
# This keeps your smoothing idea but makes it consistent with a distributional estimate.
K = len(targets)
alpha = 0.5  # Jeffreys prior per class (kept)
n_eff = float(row_probs.shape[0])

posterior_mean = (p_hat * n_eff + alpha) / (n_eff + alpha * K)

# Safety: enforce valid probabilities strictly (prevents submission failure and stabilizes KL).
posterior_mean = np.nan_to_num(
    posterior_mean, nan=1.0 / K, posinf=1.0 / K, neginf=1.0 / K
)
posterior_mean = np.clip(posterior_mean, 1e-15, None)
posterior_mean = posterior_mean / posterior_mean.sum()

mean_vote_ratio = dict(zip(targets, posterior_mean.tolist()))

for target in targets:
    sub[target] = float(mean_vote_ratio[target])

vals = sub[targets].to_numpy(dtype=float)
vals = np.nan_to_num(vals, nan=0.0, posinf=0.0, neginf=0.0)
vals = np.clip(vals, 1e-15, None)
vals = vals / vals.sum(axis=1, keepdims=True)
sub[targets] = vals

try:
    display(sub.head())
except NameError:
    print(sub.head())



,eeg_id,seizure_vote,lpd_vote,gpd_vote,lrda_vote,grda_vote,other_vote
0,2578018731,0.208766,0.131486,0.124284,0.142321,0.180536,0.212606
1,2578018731,0.208766,0.131486,0.124284,0.142321,0.180536,0.212606
2,2578018731,0.208766,0.131486,0.124284,0.142321,0.180536,0.212606
3,2578018731,0.208766,0.131486,0.124284,0.142321,0.180536,0.212606
4,1706196508,0.208766,0.131486,0.124284,0.142321,0.180536,0.212606


In [8]:
sub.to_csv("/kaggle/working/submission.csv", index=False)
print("Wrote /kaggle/working/submission.csv with shape:", sub.shape)
print(
    "Row-sum check (min/max):",
    sub[targets].sum(axis=1).min(),
    sub[targets].sum(axis=1).max(),
)

Wrote /kaggle/working/submission.csv with shape: (9850, 7)
Row-sum check (min/max): 1.0 1.0
